In [ ]:
%load_ext autoreload
%autoreload 2

# %matplotlib widget

In [ ]:
%run VD-1x8x14-TPFiltering.ipynb

In [ ]:
from tpvalidator.utils import fieldswapper

In [ ]:
from tpvalidator.analysis.tpfilter import SOTFilterAnalyser

## Create analyzer object

In [ ]:
dfa = SOTFilterAnalyser(rad_ws, em_ws)

In [ ]:
from tpvalidator.viz.textual import dataframe_to_rich_table

## Calculate TP filter efficiency on backtracked Ar39 TPs as a function of SOT cut

- The reduction in number of TPs of the sample is used to calculate the filter efficiency

In [ ]:
print(dfa.make_sot_bkg_eff_table(rop=2))
fig = dfa.plot_bkg_tpcounts_efficiency(rop=2)


# TP filter efficiency on backtracked $e^{-}$ TPs as a function of SOT cut

- Efficienct based on both number of TPs and adc_integral sum

## Signal Efficiency - Number of TPs & ADCSum

Plotting signal filtering efficiency in bins of energy of the single electron using 2 different metrics:
- Number of trigger primitives
- Total collected charge

Both quantities are calculated at dataset level

In [ ]:
fig, axes = plt.subplots(1,2, figsize=(12,5))

ax=axes[0]

# Temporarily set 'sig_weight' to None
with fieldswapper(dfa, 'sig_weight', None) as dfa_alt:
    dfa_alt.plot_signal_sot_tpfilt_counts_ke_matrix(weight=None,ax=ax)
    ax.set_title("TP Count efficiency")

ax=axes[1]
dfa.plot_signal_sot_tpfilt_counts_ke_matrix(ax=ax)
ax.set_title("ADCSum efficiency")

fig.tight_layout()


## Observations

---

# ROC curves

Display the combined effect of the min `samples_over_threshold` cuts on Ar39 and $e^-$ particle gun events

- Ar39 TPs rejection efficiency vs $e^-$ acceptance as a function of the electron kinetic energy 

## Build radiological + $e^-$ efficiency dataset ($e^-$ efficiency in  KE bins)

In [ ]:
sig_weight=None
sig_weight='adc_integral'
rop=2j
query='adc_peak> 45'
generator_sel='Ar39GenInLAr'



# df_eff = dfa.make_tpfilt_efficiency_by_ke_df(dfa.var, dfa.var_cuts, dfa.ke_bins, bkg_weight=dfa.bkg_weight, sig_weight=dfa.sig_weight, generator_sel=dfa.bkg_generator_sel, query=dfa.tp_query)
df_eff = dfa.make_sot_tpfilt_efficiency_by_ke_df(rop=2)

df_eff


## ROC - KE matrix
- Matrix of ROC curves in bins of KE

In [ ]:
sig_weight='adc_integral'
rop=2
generator_sel='Ar39GenInLAr'
refcuts = [8.5, 9.5,10.5]

fig = dfa.plot_sot_tpfilt_roc_matrix(rop=rop, refcuts=refcuts)


## ROCs, by plane and energy range

In [ ]:

fig, axes = plt.subplots(1, 3, figsize=(16,5))
for r in range(0,3):
    dfa.plot_signal_sot_tpfilt_eff_by_ke(rop=r, ax=axes[r])
    axes[r].set_title(f"Plane {r}")


### Observations
- The collection planes ROC (2) is the sharpest.
- Plane 1 has the smallest AOC at all energies
  - This is compatible with the Plane 1 signal being the least pronounced
---

## Event filtering efficiencies

The following part of the notebook uses event-based filtering to calculate the efficiency.

This is expected to be equivalent to TP-level filtering, for a SOT-based filter, but it is useful to 
- Confirm the TP filter results
- See the impact of the cut on a per-event basis

In [ ]:
with fieldswapper(dfa, 'sig_weight', None) as dfa_alt:
    tpcounts_ev = dfa_alt.make_sig_sot_evfilt_counts_vs_ke_df()

acdsum_ev = dfa_alt.make_sig_sot_evfilt_counts_vs_ke_df()

display(tpcounts_ev.loc[:5])
display(acdsum_ev.loc[:5])

In [ ]:
rop = 2
plt_cols = ['no_samples_over_threshold_cut', 'samples_over_threshold_min_8.5', 'samples_over_threshold_min_9.5', 'samples_over_threshold_min_10.5']


In [ ]:


fig, ax = plt.subplots()

cmap = plt.get_cmap('tab10')

for i, c in enumerate(plt_cols):
    tpcounts_ev.query(f'readout_plane_id =={rop}').plot.scatter(x='kinetic_energy', y=c, ax=ax, s=1, c=[cmap(i)], label=c)

ax.grid()
ax.legend()
ax.set_ylabel('counts')

fig.tight_layout()


In [ ]:

fig, ax = plt.subplots()

cmap = plt.get_cmap('tab10')

for i, c in enumerate(plt_cols):
    acdsum_ev.query(f'readout_plane_id == {rop}').plot.scatter(x='kinetic_energy', y=c, ax=ax, s=1, c=[cmap(i)], label=c)

ax.legend()


ax.grid()
ax.legend()
ax.set_ylabel('ADC Sum (event)')

fig.tight_layout()


In [ ]:
fig, ax = plt.subplots(figsize=(10,8))

cmap = plt.get_cmap('tab10')


adc_sum_ev_rop = acdsum_ev.query(f'readout_plane_id == {rop}')

eff_adcsum_ev_rop = adc_sum_ev_rop[['kinetic_energy']]

for i, c in enumerate(plt_cols):
    eff_adcsum_ev_rop[c] = adc_sum_ev_rop[c]/adc_sum_ev_rop['no_samples_over_threshold_cut']


for i, c in enumerate(plt_cols):
    eff_adcsum_ev_rop.plot.scatter(x='kinetic_energy', y=c, ax=ax, s=1, c=[cmap(i)], label=c)
    
ax.grid()
ax.legend()
ax.set_xlabel('Kinetic Energy')
ax.set_ylabel('ADC Sum (event)')

fig.tight_layout()

## e-minus roc based on single-event efficiency

In [ ]:
from tpvalidator.viz.efficiency import plot_roc
rop =2

# ke_h = hist.Hist(ke_axis)
#---------------------
# 
# Calculate counts per event
acdsum_ev = dfa.make_sig_evfilt_counts_vs_ke_df(dfa.var, query=dfa.tp_query, weight=dfa.sig_weight, var_cuts=dfa.var_cuts)

# Extact one plane
adc_sum_ev_rop = acdsum_ev.query(f'readout_plane_id == {rop}')

# Create a new dataframe, ke only
eff_adcsum_ev_rop = adc_sum_ev_rop[['event_uid', 'kinetic_energy']].copy()


sot_cols = [c for c in adc_sum_ev_rop.columns if c.startswith('samples_over_threshold_min')]
# Calculate the event efficiencies
for i, c in enumerate(sot_cols):
    eff_adcsum_ev_rop[c] = adc_sum_ev_rop[c]/adc_sum_ev_rop['no_samples_over_threshold_cut']
#---------------------


# Create the kinetic energy axis
# ke_axis = hist.axis.Regular(10,0,100/1000, underflow=True, overflow=True, name='kinetic_energy')
ke_axis = hist.axis.Variable([e/1000 for e in dfa.ke_edges], underflow=True, overflow=True, name='kinetic_energy')

# Print the events
display(eff_adcsum_ev_rop)

# Extract the list of "min_sot" columns 
cols = [c for c in eff_adcsum_ev_rop.columns if c.startswith('samples_over_threshold_min_')]

# Create an "mean" historgram for each sot_min value
h_ke_effs = {}
for c in cols:
    h_ke_effs[c] = hist.Hist(ke_axis, storage=hist.storage.Mean()).fill(kinetic_energy=eff_adcsum_ev_rop['kinetic_energy'], sample=eff_adcsum_ev_rop[c])



## Build the filter efficiency dataset (bkg and signal in KE bins)

In [ ]:
import re

rows = []
for n, h in h_ke_effs.items():
    m = re.match(r'samples_over_threshold_min_(([0-9]*[.])?[0-9]+)', n)
    if m is None:
        raise KeyError(f"Unexpected histogram {n} (name doesn't start with samples_over_threshold_min_)")
    
    sot_min = float(m.group(1))

    a  = h.axes[0]
    row = {'samples_over_threshold_min': sot_min}
    for i in range(a.size):
        lo, hi = a.bin(i)

        v = h[i]

        c = f'ev_eff_sig_{int(lo*1000)}_to_{int(hi*1000)}'
        row[c] = v.value

        c = f'err_ev_eff_sig_{int(lo*1000)}_to_{int(hi*1000)}'
        row[c] = v.variance


    rows.append(row)


sig_evfilt_eff = pd.DataFrame(rows)

# Calculate background TP filtering efficiency
# Note: "event efficiency" doesn't mean much for backgrounds
bkg_tpfilt_eff = dfa.make_tpfilt_bkg_efficiency_df(dfa.var, dfa.var_cuts, generator_sel=dfa.bkg_generator_sel, query=dfa.tp_query)
bkg_tpfilt_eff.rename(columns={'tp_eff':'tp_bkg_eff', 'err_tp_eff':'err_tp_bkg_eff'}, inplace=True)


ev_eff = bkg_tpfilt_eff.merge(sig_evfilt_eff, on='samples_over_threshold_min')
ev_eff.set_index('samples_over_threshold_min', inplace=True)


display(ev_eff)


In [ ]:
from tpvalidator.utils import subplots_autogrid

refcuts=[8.5, 9.5,10.5]
sig_eff_cols = [c for c in ev_eff.columns if c.startswith('ev_eff')]

fig, axes = subplots_autogrid(len(sig_eff_cols), figsize=(16,12))

for i, sc in enumerate(sig_eff_cols):
    plot_roc(ev_eff, 'tp_bkg_eff', sc, ax=axes[i], refcuts=refcuts, xlabel='Ar39 efficiency', ylabel='e-minus efficiency')
    axes[i].set_title(sc)

fig.tight_layout()


---

# Equivalent $E_{dep}$ cut

In [ ]:
fig, axes = plt.subplots(figsize=(8,6)) 
ax = axes


bins=np.linspace(0,6000,100)

sot_cut = 8.5
adc_peak_cut = 45
var = 'adc_integral'
ws = em_ws
# bins=np.linspace()
ws.tps.adc_integral.hist(bins=bins, histtype='step', label='all')
for t in np.arange(0.3, 0.6, 0.1):
    ws.tps.query(f'bt_edep > {t}')['adc_integral'].hist(bins=bins, histtype='step', linestyle='-.', label='$E_{dep}$ > '+f'{t:.2}')
ws.tps.query(f'samples_over_threshold > {sot_cut}')['adc_integral'].hist(bins=bins, histtype='step', label=f'sot >= {sot_cut}',color='k')
sot_cut = 9.5
ws.tps.query(f'samples_over_threshold > {sot_cut}')['adc_integral'].hist(bins=bins, histtype='step', label=f'sot >= {sot_cut}',color='dimgrey')
sot_cut = 10.5
ws.tps.query(f'samples_over_threshold > {sot_cut} & adc_peak > {adc_peak_cut}')['adc_integral'].hist(bins=bins, histtype='step', label=f'sot >= {sot_cut} + adc_peak > {adc_peak_cut}',color='purple')

ax.set_xlabel('adc integral')
ax.set_ylabel('counts')
ax.set_title('e-minus')
ax.legend()
ax.set_yscale('log')

fig.tight_layout()

In [ ]:
print(rad_tpp.make_generator_activity_table('adc_peak >= 45 & samples_over_threshold > 5.5', norm='rate', geo_norm='crp'))
print(rad_tpp.make_generator_activity_table('adc_peak >= 45 & samples_over_threshold > 6.5', norm='rate', geo_norm='crp'))
print(rad_tpp.make_generator_activity_table('adc_peak >= 45 & samples_over_threshold > 7.5', norm='rate', geo_norm='crp'))
print(rad_tpp.make_generator_activity_table('adc_peak >= 45 & samples_over_threshold > 8.5', norm='rate', geo_norm='crp'))
print(rad_tpp.make_generator_activity_table('adc_peak >= 45 & samples_over_threshold > 9.5', norm='rate', geo_norm='crp'))
